# GeoPandas AI - Analysis of the Retry Mechanism and Template sizes

In this Jupyter Notebook, we propose to analysise the usefulness of the retry mechanism and its impact on templates sizes using GeoPandas AI on the Flood dataset.

It is important to note that we will not verify whether GeoPandasAI is producing the output desired for the user, but only the frequency at which GeoPandasAI performs a retry operation. 
As a reminder, the aim of the retry operation is to detect if the LLM produced an invalid code and ask it to repair it given the error message. It is not aimed at guiding the LLM when its code is not answering the requested query.


## Initialisation of GeoPandas AI

GeoPandas AI relies on an external AI service, which you can configure through LiteLLM.

In [1]:
import os
import json
os.environ["VERTEX_CREDENTIALS"] =  json.dumps(json.load(open("/home/gilles/google-credentials.json")))

In [2]:
import os

import geopandas as gpd
import geopandasai as gai
from folium import Map
from geopandasai import update_geopandasai_config
from matplotlib.figure import Figure


update_geopandasai_config(
    lite_llm_config={
        "model": "vertex_ai/gemini-2.0-flash", # You can change this according to litellm documentation.
        "temperature": 0,  # To be albe to replicate this tutorial, not recommended for real usage
        "vertex_credentials": os.environ.get("VERTEX_CREDENTIALS"),
    }
)

A temperature of 0 is provided to increase the reproductibility of this notebook. However, this is not enough to guarentee that the LLM will produce 100% deterministic results. Therefore, resetting the cache may lead to different outputs.

In [3]:
gai.reset_cache()

## Generation of random queries

In order evaluate how often the retry mechanism is called and how often it actually allows to repair failing code, we first ask the LLM to generate random queries.

In [4]:
from litellm import completion

queries = 50

prompt = "I am evaluating an llm for GeoSpatial Data analysis. In need you to generate queries in natural languages to ask to the LLM.\
the queries must be related to one of the three following datasets:\
- floodedAreas: contains a flooded area;\
- highways: contains roads;\
- facilities: contains a list of facilities with their locations (polygone)\
Please include queries on proximity, plotting, containement and simple statistics. \
Please write me the python code so that I can try these queries directly. Queries are called using:\
dataset.chat(query, [additional dataset])\
For instance:\
floodedAreas.chat('Which higways are flooded', highways).\
Please provide me "+ str(queries)+" of such examples (the function call on the dataset not just the query) in a python list of strings that I will execute.\
Just print me the python code such as in the example, not other explainations.\
"

response = completion(
    model="vertex_ai/gemini-2.0-flash",
    messages=[{"role": "user", "content": prompt}],
    temperature=0.0,           
    vertex_credentials=os.environ.get("VERTEX_CREDENTIALS"),
)
answer = response["choices"][0]["message"]["content"]
print(answer)

```python
queries = [
    "floodedAreas.chat('What is the total area flooded?', floodedAreas)",
    "highways.chat('What is the total length of highways?', highways)",
    "facilities.chat('How many facilities are there?', facilities)",
    "floodedAreas.chat('Show me the flooded area on a map.', floodedAreas)",
    "highways.chat('Plot all highways on a map.', highways)",
    "facilities.chat('Plot all facilities on a map.', facilities)",
    "floodedAreas.chat('Which facilities are located within the flooded area?', [facilities])",
    "highways.chat('Which highways are closest to the flooded area?', [floodedAreas])",
    "facilities.chat('Which facilities are within 1km of a highway?', [highways])",
    "floodedAreas.chat('What is the average depth of the flood?', floodedAreas)",
    "highways.chat('What is the most common type of highway?', highways)",
    "facilities.chat('What types of facilities are there?', facilities)",
    "floodedAreas.chat('How many highways are affected by

## Patching of GeoPandasAI

In order not to add additional code to the library itsel, the concerned method is patched to add the logic required to collect the desired statistics.

In [5]:
old_build_code = gai.services.code._internal.code.build_code

import geopandasai.services.code._internal.code as code_module
import geopandasai.services.code._internal.magic as magic_module
from typing import List, Type, Union
import traceback

import matplotlib.pyplot as plt

from geopandasai.services.code._internal.execute import execute_func
from geopandasai.services.code._internal.samples import SAMPLES
from geopandasai.services.code.template import (
    prompt_with_template,
    parse_template,
    Template,
)
from geopandasai.services.description import describe_dataframe
from geopandasai.shared.constants import FUNCTION_SIGNATURE
from geopandasai.shared.return_type import type_to_literal
from geopandasai.shared.types import GeoOrDataFrame

from geopandasai.services.code._internal.code import build_static_description, dfs_to_string



def new_build_code(
    prompt: str,
    return_type: Type,
    dfs: List[GeoOrDataFrame],
    history: str = None,
    user_provided_libraries: List[str] = None,
) -> Union[str, None]:
    dataset_description, libraries_str, system_instructions = build_static_description(
        dfs, user_provided_libraries
    )
    dfs_string = ", ".join([f"df_{i + 1}" for i in range(len(dfs))])

    history = history or "N/A"
    
    if not hasattr(new_build_code, "success"):
        new_build_code.success = 0
        new_build_code.success_after_retry = []
        new_build_code.fails = 0
        new_build_code.template_sizes = []
    
    max_attempts = 5
    last_code = None
    last_exception = None
    response = None

    for attempt in range(max_attempts):
        print(attempt, end=" ")
        if last_code:
            template = parse_template(
                Template.CODE_PREVIOUSLY_ERROR,
                system_instructions=system_instructions,
                last_code=last_code,
                last_exception=last_exception,
                libraries=libraries_str,
                prompt=prompt,
                history=history,
                return_type=type_to_literal(return_type),
                dfs=dfs_string,
                dataset_description=dataset_description,
                tips=SAMPLES,
                function_signature=FUNCTION_SIGNATURE,
            )
            if attempt == 0:
                new_build_code.template_sizes.append([template])
            else:
                new_build_code.template_sizes[-1].append(template)
            last_code = prompt_with_template(
                template, remove_markdown_code_limiter=True
            )
        else:
            template = parse_template(
                Template.CODE,
                system_instructions=system_instructions,
                last_code=last_code,
                last_exception=last_exception,
                libraries=libraries_str,
                history=history,
                prompt=prompt,
                dfs=dfs_string,
                return_type=type_to_literal(return_type),
                dataset_description=dataset_description,
                tips=SAMPLES,
                function_signature=FUNCTION_SIGNATURE,
            )
            if attempt == 0:
                new_build_code.template_sizes.append([template])
            else:
                new_build_code.template_sizes[-1].append(template)
            last_code = prompt_with_template(
                template, remove_markdown_code_limiter=True
            )

            last_code = prompt_with_template(
                template, remove_markdown_code_limiter=True
            )
        try:
            execute_func(last_code, return_type, *dfs)
            response = last_code
            new_build_code.success += 1
            if attempt > 0:
                new_build_code.success_after_retry.append(attempt)
            break
        except Exception as e:
            if attempt == max_attempts -1:
                new_build_code.fails += 1
            last_exception = f"{str(e)}\n{traceback.format_exc()}"

    # clear matplotlib cache to avoid memory issues
    plt.close("all")

    if not response:
        new_build.code.fails += 1
        raise ValueError(
            "No valid code snippet. Here is the last code snippet that was generated:\n"
            f"{last_code}\n\nAnd the last exception that was raised:\n{last_exception}"
        )

    return response

gai.services.code._internal.code.build_code = new_build_code
code_module.build_code = new_build_code
magic_module.build_code = new_build_code

## Retry and Token Analysis

The exact number of tokens used is dependant of the LLM model and tokenizer. As the tokenizer for gemini-2.0-flash is not public, we will repport the number of tokens used by the tokenizer of the gemini-1.5-flash-002 model.

In [6]:
floodedAreas_gdf = gpd.read_file("./Data/Shps/boiseFlood200yr_v4.shp") # gai.read_file(...), would also work and directly instantiate a GeoDataFrameAI
highways_gdf = gpd.read_file("./Data/Shps/primaryHighways.gpkg")
facilities_gdf = gpd.read_file('./Data/Shps/schoolsHospitalsPoliceFire.gpkg')

In [7]:
floodedAreas = gai.GeoDataFrameAI(floodedAreas_gdf)
highways = gai.GeoDataFrameAI(highways_gdf, description="This dataset contains information about roads.")
facilities = gai.GeoDataFrameAI(facilities_gdf)

/home/gilles/.local/lib/python3.10/site-packages/geopandas/geodataframe.py:223: UserWarning: Pandas doesn't allow columns to be created via a new attribute name - see https://pandas.pydata.org/pandas-docs/stable/indexing.html#attribute-access
  super().__setattr__(attr, val)


In [8]:
queries = [
    "floodedAreas.chat('What is the total area flooded?', floodedAreas)",
    "highways.chat('What is the total length of highways?', highways)",
    "facilities.chat('How many facilities are there?', facilities)",
    "floodedAreas.chat('Show me the flooded area on a map.', floodedAreas)",
    "highways.chat('Plot all highways on a map.', highways)",
    "facilities.chat('Plot all facilities on a map.', facilities)",
    "floodedAreas.chat('Which facilities are located within the flooded area?', [facilities])",
    "highways.chat('Which highways are closest to the flooded area?', [floodedAreas])",
    "facilities.chat('Which facilities are within 1km of a highway?', [highways])",
    "floodedAreas.chat('What is the average depth of the flood?', floodedAreas)",
    "highways.chat('What is the most common type of highway?', highways)",
    "facilities.chat('What types of facilities are there?', facilities)",
    "floodedAreas.chat('How many highways are affected by the flood?', [highways])",
    "highways.chat('Show me the highways that intersect the flooded area.', [floodedAreas])",
    "facilities.chat('Which facilities are most vulnerable to flooding?', [floodedAreas])",
    "floodedAreas.chat('What is the perimeter of the flooded area?', floodedAreas)",
    "highways.chat('What is the longest highway?', highways)",
    "facilities.chat('Which facility has the largest area?', facilities)",
    "floodedAreas.chat('Plot the flooded area and nearby highways on a map.', [highways])",
    "highways.chat('Plot the highways and nearby facilities on a map.', [facilities])",
    "facilities.chat('Plot the facilities and the flooded area on a map.', [floodedAreas])",
    "floodedAreas.chat('Which highways are completely submerged?', [highways])",
    "highways.chat('Which highways are partially submerged?', [floodedAreas])",
    "facilities.chat('Which facilities are inaccessible due to flooding?', [floodedAreas, highways])",
    "floodedAreas.chat('What is the maximum depth of the flood?', floodedAreas)",
    "highways.chat('What is the average speed limit on the highways?', highways)",
    "facilities.chat('What is the average distance between facilities?', facilities)",
    "floodedAreas.chat('Show me the flooded area with a buffer of 500m.', floodedAreas)",
    "highways.chat('Show me the highways with a buffer of 100m.', highways)",
    "facilities.chat('Show me the facilities with a buffer of 200m.', facilities)",
    "floodedAreas.chat('Which facilities are within the 500m buffer of the flooded area?', [facilities])",
    "highways.chat('Which facilities are within 100m of a highway?', [facilities])",
    "facilities.chat('Which highways are within 200m of a facility?', [highways])",
    "floodedAreas.chat('What is the percentage of highways affected by the flood?', [highways])",
    "highways.chat('What is the percentage of highways that are major roads?', highways)",
    "facilities.chat('What is the percentage of facilities that are hospitals?', facilities)",
    "floodedAreas.chat('Show me the flooded area and the closest hospital.', [facilities])",
    "highways.chat('Show me the highway with the highest traffic volume.', [facilities])",
    "facilities.chat('Show me the facility with the highest number of employees.', facilities)",
    "floodedAreas.chat('What is the impact of the flood on transportation?', [highways])",
    "highways.chat('What is the impact of the flood on access to facilities?', [floodedAreas, facilities])",
    "facilities.chat('What is the impact of the flood on the operation of facilities?', [floodedAreas])",
    "floodedAreas.chat('What is the estimated cost of damage caused by the flood?', floodedAreas)",
    "highways.chat('What is the estimated cost of repairing the damaged highways?', [floodedAreas])",
    "facilities.chat('What is the estimated cost of repairing the damaged facilities?', [floodedAreas])",
    "floodedAreas.chat('Which areas are most prone to flooding?', floodedAreas)",
    "highways.chat('Which highways are most vulnerable to flooding?', highways)",
    "facilities.chat('Which facilities are most critical during a flood?', facilities)",
    "floodedAreas.chat('Suggest evacuation routes based on the flooded area.', [highways, facilities])",
    "highways.chat('Suggest alternative routes to bypass the flooded area.', [floodedAreas])"
]

In [9]:
gai.reset_cache()
for i, query in enumerate(queries):
    try:    
        print(i, query, end=", ")
        exec(query)
        print()
    except Exception as e:
        print("Error:")
        
print(new_build_code.success, new_build_code.fails, new_build_code.success_after_retry)


0 floodedAreas.chat('What is the total area flooded?', floodedAreas), 0 
1 highways.chat('What is the total length of highways?', highways), 0 
2 facilities.chat('How many facilities are there?', facilities), 0 
3 floodedAreas.chat('Show me the flooded area on a map.', floodedAreas), 0 
4 highways.chat('Plot all highways on a map.', highways), 0 
5 facilities.chat('Plot all facilities on a map.', facilities), 0 
6 floodedAreas.chat('Which facilities are located within the flooded area?', [facilities]), 0 
7 highways.chat('Which highways are closest to the flooded area?', [floodedAreas]), 0 1 2 
8 facilities.chat('Which facilities are within 1km of a highway?', [highways]), 0 1 2 3 4 Error:
9 floodedAreas.chat('What is the average depth of the flood?', floodedAreas), 0 
10 highways.chat('What is the most common type of highway?', highways), 0 
11 facilities.chat('What types of facilities are there?', facilities), 0 
12 floodedAreas.chat('How many highways are affected by the flood?', [h

The retry mechanism was called on 13 different queries (26%). Out of the 13 queries, the retry mechanism allowed to repair and obtain a working code for 10 of them (77%).

In [11]:
from vertexai.preview import tokenization

model_name = "gemini-1.5-flash-002"  # one of the supported models, 
tokenizer = tokenization.get_tokenizer_for_model(model_name)

print(f"Query, Template lenghs, Approximate number of Tokens")

lenghts = []
tokens = []
for i, query_templates in enumerate(new_build_code.template_sizes):
    temp_lenghts = [len(str(t)) for t in query_templates]
    temp_tokens = [tokenizer.count_tokens(str(t)).total_tokens for t in query_templates]
    print(i, temp_lenghts, temp_tokens)
    lenghts.append(temp_lenghts)
    tokens.append(temp_tokens)

    
print("Average number of tokens used by query:", sum([sum(t) for t in tokens])/len(tokens))
print("Max number of tokens for a query:", max([sum(t) for t in tokens]))

Query, Template lenghs, Approximate number of Tokens
0 [16768] [5097]
1 [46064] [12492]
2 [22067] [7860]
3 [16795] [5109]
4 [46078] [12501]
5 [22094] [7871]
6 [15443] [4429]
7 [30082, 19174, 20470] [8126, 5330, 5724]
8 [18149, 8063, 9103, 9103, 9261] [5831, 3319, 3710, 3708, 3754]
9 [16776] [5099]
10 [46063] [12493]
11 [22074] [7861]
12 [15432] [4429]
13 [30088, 19253] [8127, 5364]
14 [18091] [5810]
15 [16779] [5099]
16 [46051] [12490]
17 [22073, 11445] [7861, 5133]
18 [15467, 4884, 4963, 4981, 4981] [4441, 1764, 1791, 1793, 1795]
19 [30110] [8137]
20 [18118, 7926] [5823, 3287]
21 [15430] [4426]
22 [30074] [8123]
23 [18092] [5810]
24 [16776, 5372] [5099, 2162]
25 [46071] [12494]
26 [22089, 13419] [7862, 5794]
27 [16808] [5115]
28 [42550] [11572]
29 [23206] [9250]
30 [15454, 4713, 5547, 5492, 6004] [4436, 1669, 1988, 1985, 2111]
31 [28353, 18921] [7676, 5422]
32 [18612, 7904, 9894, 10005] [6576, 3864, 4484, 4519]
33 [15449] [4431]
34 [42503] [11551]
35 [23147] [9386]
36 [15466, 5041, 60